In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.formula.api import logit
from scipy.stats import chi2_contingency
import statsmodels.api as sm

from pathlib import Path
import pickle
import os

#### Project Question
* Does region, payment method, age, or gender also predict customer churn?

In [2]:
# Import dataset
root = Path.cwd().parent
dir_path = Path.cwd()
dataset = root/'data'/'customer_churn_clean.csv'
results_dir = f'{dir_path}/results/'

df = pd.read_csv(dataset, index_col=0)
df.head()

,customer_id,gender,region,subscription_plan,payment_method,age,days_since_last_login,customer_service_calls,monthly_spend,churn
0,10001,Male,East,Basic,PayPal,27.0,27,1.0,61.08,0
1,10002,NaN,West,Pro,PayPal,55.0,146,2.0,73.09,0
2,10003,Female,West,Pro,E-Wallet,NaN,147,2.0,116.30,1
3,10004,Female,South,Basic,Credit Card,NaN,34,1.0,0.58,1
4,10005,Male,North,Basic,Crypto,30.0,146,3.0,46.35,1


In [3]:
# Understanding data columns, dtypes, and observation count 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             10000 non-null  int64  
 1   gender                  9039 non-null   str    
 2   region                  10000 non-null  str    
 3   subscription_plan       10000 non-null  str    
 4   payment_method          9300 non-null   str    
 5   age                     8491 non-null   float64
 6   days_since_last_login   10000 non-null  int64  
 7   customer_service_calls  9950 non-null   float64
 8   monthly_spend           9441 non-null   float64
 9   churn                   10000 non-null  int64  
dtypes: float64(3), int64(3), str(4)
memory usage: 781.4 KB


#### Analysis Variables
* Region
    - Geographic territory
* Gender
    - Reported customer gender
* Payment Method
    - Most frequent payment type
* Age
    - Customer's age in years

**_Objectives_**
* Identify the relationship between analysis variables and churn
* Perform chi-squared test of independece to all variables to determine if any have a statistically significant relationship with churn
* Conduct an isolated and logistic inferential logistic regression test on significant variables in relation to churn

In [4]:
# Subsetting the dataset
subset = df[['region','gender','payment_method','age','churn']].dropna().copy()
subset.age = subset.age.astype(int)
subset.head()

,region,gender,payment_method,age,churn
0,East,Male,PayPal,27,0
4,North,Male,Crypto,30,1
5,West,Male,Crypto,46,1
6,South,Male,E-Wallet,22,1
7,East,Non-Binary,E-Wallet,44,1


**_Conducting Chi-Squared Test of Independence_**

$H_0:$ There is no association between customer churn and region

$H_1:$ There is an association between customer churn and region

In [5]:
def save_result(path:str,filename:str,data):
    with open(path+filename+'.pkl', 'wb') as to_write:
        pickle.dump(data,to_write)

def read_result(path:str,filename:str):
    with open(path+filename+'.pkl', 'rb') as to_read:
        results_data = pickle.load(to_read)
    return results_data

def multi_chi_test(columns,data,target,alpha=0.05,results_dir=None,test_name=None):
    rows = []
    for col in columns:
        observed = pd.crosstab(data[target],data[col])
        res = chi2_contingency(observed)

        rows.append({
            'Model':f'{target} ~ {col}',
            'Chi2 Value':(res.statistic).round(2),
            'Significance Level':alpha,
            'P-value':res.pvalue,
            'Significant':res.pvalue<alpha,
            'Degrees of Freedom':res.dof
        })
    results = pd.DataFrame(rows)

    if results_dir and test_name:
        save_result(results_dir,test_name,results)
results = read_result(results_dir,'chi_square_test_results').reset_index(drop=True)
results

,Model,Chi2 Value,Significance Level,P-value,Significant,Degrees of Freedom
0,Churn ~ region,31.65,0.05,0.00000062,True,3
1,Churn ~ gender,0.07,0.05,0.96620874,False,2
2,Churn ~ payment_method,4.65,0.05,0.19904529,False,3
3,Churn ~ age,57.34,0.05,0.22158986,False,50


#### Chi-Squared Test on Analysis Variables
* Results shows that region have a statistically significant relationship with customer churn with a p-value of `6.2e-7`, less than the significance level of `0.05`.

**_Next Steps_**
* Inferential Logistic Regression Analysis of `Churn ~ region` to determine if the results still hold across all region

**_Inferential Isolated Logistic Regression Analysis_**

_Variables:_
* Churn
* Region

$H_0:$  There is no statistical significant relationship between region and churn
        Churn rate across all region are the same

$H_1:$  There is a statistical significant relationship between region and churn
        Churn rate across all regions differ significantly

In [6]:
def logit_model(formula, data):
    model = logit(formula,data).fit()
    return model

# Conducting Inferential Isolated Logistic Regression
# Variables: churn, region
formula = 'churn ~ C(region)'
model = logit_model(formula, subset)
model.summary().tables[1]

Optimization terminated successfully.
         Current function value: 0.549030
         Iterations 5


,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,0.9282,0.053,17.606,0.000,0.825,1.032
C(region)[T.North],0.1955,0.076,2.559,0.010,0.046,0.345
C(region)[T.South],0.4207,0.078,5.365,0.000,0.267,0.574
C(region)[T.West],0.2985,0.077,3.862,0.000,0.147,0.450


In [7]:
params = model.params.copy()
params.index = (
    model.params.index
    .str.replace(r'C\(([^)]+)\)', r'\1', regex=True)
)
params.index

Index(['Intercept', 'region[T.North]', 'region[T.South]', 'region[T.West]'],
      dtype='str')

In [8]:
def get_odds(model):
    intercept = model.params.iloc[0]
    params = model.params.drop('Intercept')
    
    inter_row=[{
        'Row':'Region: East',
        'Odds Ratio':np.exp(intercept)
    }]

    for row in params.index:
        inter_row.append({
            'Row':row,
            'Odds Ratio':np.exp(model.params[row]+intercept)
        })

    return pd.DataFrame(inter_row).reset_index(drop=True)

def name_clean(model):
    return model.str.replace(
        r'C\(([^)]+)\)\[T\.(.*?)\]',
        lambda m: f'{m.group(1).capitalize()}: {m.group(2).capitalize()}',
        regex=True)

odds_model = get_odds(model)
odds_model.Row = (name_clean(odds_model.Row))
odds_model


,Row,Odds Ratio
0,Region: East,2.529880
1,Region: North,3.076037
2,Region: South,3.852941
3,Region: West,3.409877


**_Results_**
* Results shows that all regions have a p-value less than the significance level of `0.05`, rejecting the null hypothesis and conclude that churn rate across all regions differ significantly.
* In contrast to `East` region, 

In [9]:
# Conducting Inferential Controlled Logistic Regression
# Variables: churn, region, spend, customer service calls, days since last login
# Region is tested while controlling for specified variables that is statistically significant with churn

subset1 = df[['region','gender','payment_method','age','monthly_spend','customer_service_calls','days_since_last_login','churn']].dropna().copy()
subset1.customer_service_calls = subset1.customer_service_calls.clip(upper=6)
formula = 'churn ~ C(region) + monthly_spend + C(customer_service_calls) + days_since_last_login'
inf_model = logit_model(formula, subset1)
inf_model.summary().tables[1]

Optimization terminated successfully.
         Current function value: 0.479910
         Iterations 7


,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,-0.3137,0.104,-3.009,0.003,-0.518,-0.109
C(region)[T.North],0.2327,0.086,2.712,0.007,0.065,0.401
C(region)[T.South],0.5090,0.087,5.842,0.000,0.338,0.680
C(region)[T.West],0.3136,0.086,3.633,0.000,0.144,0.483
C(customer_service_calls)[T.1.0],0.4211,0.086,4.886,0.000,0.252,0.590
C(customer_service_calls)[T.2.0],0.7415,0.090,8.235,0.000,0.565,0.918
C(customer_service_calls)[T.3.0],1.1531,0.108,10.652,0.000,0.941,1.365
C(customer_service_calls)[T.4.0],1.3725,0.150,9.145,0.000,1.078,1.667
C(customer_service_calls)[T.5.0],2.3445,0.312,7.511,0.000,1.733,2.956
C(customer_service_calls)[T.6.0],2.6198,0.610,4.292,0.000,1.423,3.816


**_Inferential Logistic Regression_**
* Results shows that region have a statistically significant relationship with churn on both isolated and controlled tests, having p-value of less than the significance level (`0.05`) across all regions